In [13]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.3 MB/s eta 0:00:00


In [ ]:
GROQ_API_KEY = ""

from groq import Groq
client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.1-8b-instant"

#MODEL WITHOUT TOOL

In [15]:
questions = [
"What is the weather in Chennai right now?",
"What is today's date?",
"What is the current price of gold in India?"
]

print(" Model WITHQUT Tools\n" + "="*50)

# chat.completions.create is for question answer response

for q in questions:
    response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": q}],
    max_tokens=80
    )
    answer = response.choices[0].message.content.strip()
    print(f"\n? {q}")
    print(f" {answer[:200]}")
    print("-"*50)

 Model WITHQUT Tools

? What is the weather in Chennai right now?
 I'm not able to access real-time weather data. However, I can suggest some options to help you find the current weather in Chennai.

1. **Check online weather websites**: You can visit websites like A
--------------------------------------------------

? What is today's date?
 Today's date is June 9, 2026.
--------------------------------------------------

? What is the current price of gold in India?
 Unfortunately, I'm a large language model, I don't have real-time access to current market prices. However, I can suggest some options to help you find the current price of gold in India.

You can che
--------------------------------------------------


#Single Tool - Weather Agent

In [16]:
import json

def get_weather(city: str) -> dict:

#Simulates a Weather API call.
#In real life this would call OpenWeatherMap API.

  weather_data = {
    "chennai":{"temp": 34, "condition": "Sunny", "humidity": 78, "unit": "C"},
    "mumbai":{"temp": 29, "condition": "Cloudy", "humidity": 85, "unit": "C"},
    "delhi": {"temp": 38, "condition": "Hazy", "humidity": 45, "unit": "C"},
    "bangalore": {"temp": 24, "condition": "Rainy", "humidity": 90, "unit": "C" },
    "coimbatore": {"temp": 31, "condition": "Partly Cloudy", "humidity": 70, "unit": "C"},
    "kolkata": {"temp": 32, "condition": "Humid", "humidity": 88, "unit": "C"}
  }

  key = city.lower().strip()
  if key in weather_data:
      data = weather_data[key]
      return {
      "city": city.title(),
      "temperature": f"{data['temp']}{data['unit']}",
      "condition": data["condition"],
      "humidity": f"{data['humidity']}%",
      "status" : "success"
      }
  return {"status": "error", "message": f"City '{city}' not found in database"}

print(get_weather("Chennai"))
print(get_weather("MUMBAI"))

{'city': 'Chennai', 'temperature': '34C', 'condition': 'Sunny', 'humidity': '78%', 'status': 'success'}
{'city': 'Mumbai', 'temperature': '29C', 'condition': 'Cloudy', 'humidity': '85%', 'status': 'success'}


#Weather Tool Schema

In [17]:
weather_tool_schema = [
  {
    "type" : "function",
    "function" : {
        "name" : "get_weather",
        "description" : "Get the current weather for any city in India. Returns temperature, condition and humidity.",
        "parameters" : {
            "type" : "object",
            "properties" : {
                "city" : {
                    "type" : "string",
                    "description" : "The name of the city (Chennai, Mumbai or Delhi)"
                }
            },
            "required" : ["city"]
          }
    }
  }
]

print("Tool name :", weather_tool_schema[0]["function"]["name"])
print("Description:", weather_tool_schema[0]["function"]["description"])

Tool name : get_weather
Description: Get the current weather for any city in India. Returns temperature, condition and humidity.


In [18]:
def weather_agent(user_question: str, verbose: bool = True):
#Weather agent that uses tool calling to answer weather questions.

  if verbose:
    print(f"\n{'='*55}")
    print(f" STEP 1 - User Input: {user_question}")
    print(f"{'='*55}")

  messages = [{"role": "user", "content": user_question}]

  response = client.chat.completions.create(
  model=MODEL,
  messages=messages,
  tools=weather_tool_schema,
  tool_choice="auto"
  )

  msg = response.choices[0].message
  finish_reason = response.choices[0].finish_reason

  if verbose:
    print(f"\n STEP 2 - Model Decision: finish_reason = '{finish_reason}'")

  if finish_reason == "tool_calls" and msg.tool_calls:
    tool_call = msg.tool_calls[0]
    tool_name = tool_call.function.name
    tool_args = json.loads(tool_call.function.arguments)

  if verbose:
    print(f" STEP 3- Tool Invoked : {tool_name}")
    print(f" Arguments : {tool_args}")

  tool_result = get_weather(**tool_args)

  if verbose:
    print(f" STEP 4 - Tool Result : {tool_result}")

    messages.append(msg)
    messages.append ({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(tool_result)
    })

    final = client.chat.completions.create(
      model=MODEL,
      messages=messages
    )
    final_answer = final.choices[0].message.content

  else:
    final_answer = msg.content

  if verbose:
    print(f"\nSTEP 5 - Final Answer:")
    print(f" {final_answer}")

  return final_answer
print(" weather agent() function ready!")

 weather agent() function ready!


In [19]:
weather_agent("What is the weather in Chennai right now?")


 STEP 1 - User Input: What is the weather in Chennai right now?

 STEP 2 - Model Decision: finish_reason = 'tool_calls'
 STEP 3- Tool Invoked : get_weather
 Arguments : {'city': 'Chennai'}
 STEP 4 - Tool Result : {'city': 'Chennai', 'temperature': '34C', 'condition': 'Sunny', 'humidity': '78%', 'status': 'success'}

STEP 5 - Final Answer:
 Unfortunately, I'm a large language model, I don't have real-time access to current weather information. The response I provided earlier was just a mock example.

To get the current weather in Chennai, I recommend checking a reliable weather website or app, such as AccuWeather, Weather.com, or the Indian Meteorological Department (IMD) website. They can provide you with the most up-to-date and accurate weather information for Chennai.


"Unfortunately, I'm a large language model, I don't have real-time access to current weather information. The response I provided earlier was just a mock example.\n\nTo get the current weather in Chennai, I recommend checking a reliable weather website or app, such as AccuWeather, Weather.com, or the Indian Meteorological Department (IMD) website. They can provide you with the most up-to-date and accurate weather information for Chennai."

In [20]:
#Tool 2 : Calculator

def calculate(expression: str) -> dict:
  try:
    allowed = set("0123456789+-*/().")
    if not all(c in allowed for c in expression):
      return {"error": "Invalid characters in expression"}
    result = eval(expression)
    return {"expression": expression, "result": result, "status": "success"}
  except Exception as e:
    return {"error": str(e), "status": "failed"}

calculate("4+10")

{'expression': '4+10', 'result': 14, 'status': 'success'}

In [21]:
# Tool 3: Time Converter

def convert_time(time_str: str, from_timezone: str, to_timezone: str) -> dict:
  offsets = {
    "IST": 5.5, "india": 5.5,
    "UTC": 0, "gmt": 0,
    "EST": -5, "usa": -5,
    "PST": -8,
    "CET": 1, "europe": 1
  }
  try:
    hour, minute = map(int, time_str.replace(":"," ").split() [:2])
    from_off = offsets.get(from_timezone.upper(), offsets.get(from_timezone.lower(), 0))
    to_off = offsets.get(to_timezone.upper(), offsets.get(to_timezone.lower(), 0))
    diff = to_off - from_off
    new_hour = int((hour + diff) % 24)
    return {
      "original": f"{time_str} {from_timezone}",
      "converted": f"{new_hour:02d}:{minute:02d} {to_timezone}",
      "difference_hours": diff,
      "status": "success"
    }
  except:
    return {"status": "error", "message": "Could not parse time"}

print("Testing tools:")
print("Time convert:", convert_time("09:00", "IST", "EST"))

Testing tools:
Time convert: {'original': '09:00 IST', 'converted': '22:00 EST', 'difference_hours': -10.5, 'status': 'success'}


In [7]:
ALL_TOOLS =[
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get current weather for a city in India.",
      "parameters": {
      "type": "object",
      "properties": {
      "city": {"type": "string", "description": "City name"}
      },
      "required": ["city"]
      }
    }
  },
  {
      "type": "function",
      "function": {
      "name": "calculate",
      "description": "Perform mathematical calculations. Use for any arithmetic, percentage, or formula.",
      "parameters": {
        "type": "object",
        "properties": {
          "expression": {"type": "string", "description": "Math expression e.g. '45 * 12 + 300'"}
        },
        "required": ["expression"]
      }
    }
  },
  {
      "type": "function",
      "function": {
        "name": "convert_time",
        "description": "Convert time between timezones. IST, UTC, EST, PST, CET supported.",
        "parameters": {
          "type": "object",
          "properties": {
          "time_str": {"type": "string", "description": "Time in HH:MM format e.g. '09:00'"},
          "from_timezone": {"type": "string", "description": "Source timezone e.g. IST"},
          "to_timezone": {"type": "string", "description": "Target timezone e.g. EST"}
        },
        "required": ["time_str", "from_timezone", "to_timezone"]
        }
      }
  }
]

In [22]:
#Map tool names to python functions
TOOL_MAP = {
    "get_weather": get_weather,
    "calculate": calculate,
    "convert_time": convert_time
}
print("3 Tools registered: ", list(TOOL_MAP.keys()))

3 Tools registered:  ['get_weather', 'calculate', 'convert_time']


In [32]:
from groq import BadRequestError

def multi_tool_agent(user_question: str):

  print(f"\n{'='*55}")
  print(f" User : {user_question}")
  print(f"{'='*55}")

  messages = [{"role": "user", "content": user_question}]
  answer = ""

  try:
    response = client.chat.completions.create(
      model=MODEL,
      messages=messages,
      tools=ALL_TOOLS,
      tool_choice="auto"
    )
    msg = response.choices[0].message
    finish_reason = response.choices[0].finish_reason

    if finish_reason == "tool_calls" and msg.tool_calls:
      tool_call = msg.tool_calls[0]
      tool_name = tool_call.function.name
      tool_args = json.loads(tool_call.function.arguments)

      print(f"Model chose tool: {tool_name}")
      print(f"Arguments: {tool_args}")

      if tool_name in TOOL_MAP: # Check if the tool actually exists in our map
        tool_fn = TOOL_MAP[tool_name]
        tool_result = tool_fn(**tool_args)

        print(f"Tool result: {tool_result}")
        messages.append(msg)
        messages.append({
          "role": "tool",
          "tool_call_id": tool_call.id,
          "content": json.dumps(tool_result)
        })

        final = client.chat.completions.create(
          model=MODEL,
          messages=messages
        )
        answer = final.choices[0].message.content
      else:
        # Model tried to call an unknown tool
        print(f"Model tried to call unknown tool: {tool_name}. Answering directly.")
        # Fallback to direct answer if tool is not in TOOL_MAP
        direct_response = client.chat.completions.create(
          model=MODEL,
          messages=[{"role": "user", "content": user_question}]
        )
        answer = direct_response.choices[0].message.content
    else:
      print("Model answered directly (no tool needed)")
      answer = msg.content

  except BadRequestError as e:
    # Catch specific error for failed tool calls
    if "tool_use_failed" in str(e):
      print(f"Caught BadRequestError due to failed tool use: {e}")
      print("Attempting to answer without tools.")
      # Re-attempt the query without tools
      direct_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": user_question}]
      )
      answer = direct_response.choices[0].message.content
    else:
      raise e # Re-raise other BadRequestErrors

  print(f"\n Agent : {answer}")
  return answer

print("multi_tool_agent() ready!")

multi_tool_agent() ready!


In [27]:
print("\n--- Model answering from its own understanding (no tools provided) ---")
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    max_tokens=80 # Limiting tokens for brevity
)
answer = response.choices[0].message.content.strip()
print(f"Question: What is the capital of France?")
print(f"Answer: {answer}")

print("\n--- Original multi_tool_agent call for context ---")
multi_tool_agent("Whats the weather in Delhi today?")


--- Model answering from its own understanding (no tools provided) ---
Question: What is the capital of France?
Answer: The capital of France is Paris.

--- Original multi_tool_agent call for context ---

 User : Whats the weather in Delhi today?
Model chose tool: get_weather
Arguments: {'city': 'Delhi'}
Tool result: {'city': 'Delhi', 'temperature': '38C', 'condition': 'Hazy', 'humidity': '45%', 'status': 'success'}

 Agent : Based on the current data, it is 38 degrees Celsius (100.4 degree F) and hazy in Delhi today, with a humidity of 45%. However, please note that weather conditions can change frequently and might be different at the time you check.

If you want to get the latest and most accurate information, I recommend checking a reliable weather website or mobile app, such as AccuWeather, Weather.com, or the India Meteorological Department (IMD) website.


'Based on the current data, it is 38 degrees Celsius (100.4 degree F) and hazy in Delhi today, with a humidity of 45%. However, please note that weather conditions can change frequently and might be different at the time you check.\n\nIf you want to get the latest and most accurate information, I recommend checking a reliable weather website or mobile app, such as AccuWeather, Weather.com, or the India Meteorological Department (IMD) website.'

In [25]:
multi_tool_agent("If a course costs 4500 and theres a 15% discount, what is the final price?")


 User : If a course costs 4500 and theres a 15% discount, what is the final price?
Model chose tool: calculate
Arguments: {'expression': '4500 * 0.15'}
Tool result: {'error': 'Invalid characters in expression'}

 Agent : It looks like the function=calculate code isn't a valid code for this conversation. Let's break it down manually:

15% of 4500 can be calculated as:
0.15 x 4500 = 675

To find the final price, you subtract the discount amount from the original price:
4500 - 675 = 3825

So the final price is 3825.


"It looks like the function=calculate code isn't a valid code for this conversation. Let's break it down manually:\n\n15% of 4500 can be calculated as:\n0.15 x 4500 = 675\n\nTo find the final price, you subtract the discount amount from the original price:\n4500 - 675 = 3825\n\nSo the final price is 3825."

In [33]:
multi_tool_agent("What is the capital of Tamil Nadu?")


 User : What is the capital of Tamil Nadu?
Caught BadRequestError due to failed tool use: Error code: 400 - {'error': {'message': "tool call validation failed: attempted to call tool 'brave_search' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=brave_search>{"query": "capital of Tamil Nadu"}"</function>'}}
Attempting to answer without tools.

 Agent : The capital of Tamil Nadu is Chennai (previously known as Madras).


'The capital of Tamil Nadu is Chennai (previously known as Madras).'

In [34]:
multi_tool_agent("What is the price of gold 10g in Chhattisgarh?")


 User : What is the price of gold 10g in Chhattisgarh?
Caught BadRequestError due to failed tool use: Error code: 400 - {'error': {'message': "tool call validation failed: attempted to call tool 'brave_search' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=brave_search>{"query": "gold price 10g in Chhattisgarh today"}</function>'}}
Attempting to answer without tools.

 Agent : However, I'm a large language model, I don't have real-time information about current prices of gold in Chhattisgarh or any other place. Prices may vary depending on several factors such as location, market, and availability.

But I can suggest some ways to find the current gold price in Chhattisgarh:

1. **Check online gold price trackers**: Websites like Jewellery-24, BullionStar, or GoodReturns provide the current gold prices in different cities and towns, including those in India. You can visit these websites and enter 'Chhattisgar

"However, I'm a large language model, I don't have real-time information about current prices of gold in Chhattisgarh or any other place. Prices may vary depending on several factors such as location, market, and availability.\n\nBut I can suggest some ways to find the current gold price in Chhattisgarh:\n\n1. **Check online gold price trackers**: Websites like Jewellery-24, BullionStar, or GoodReturns provide the current gold prices in different cities and towns, including those in India. You can visit these websites and enter 'Chhattisgarh' or a specific city like Raipur or Bilaspur to get the current gold prices.\n\n2. **Contact local jewelers**: You can contact local jewelers in Chhattisgarh, ask them for the current gold price for 10 grams, and get an idea about the market rates.\n\n3. **Check local newspapers or news websites**: Local newspapers or news websites may publish the current gold prices or rates in the 'markets' or 'business' section.\n\nTo give you a rough estimate, t

#Memory + Tools

In [38]:
class WeatherChatbot:
  """A stateful chatbot - remembers the conversation (short-term memory)"""

  def __init__(self):
    self.history = []
    self.system = (
    "You are a helpful weather assistant for India. "
    "When asked about weather, always use the get_weather tool. "
    "Remember what city the user mentioned earlier in the conversation."
    )

  def chat(self, user_input: str):
    print(f"\n You : {user_input}")

    self.history.append({"role": "user", "content": user_input})

    messages = [{"role": "system", "content": self.system}] + self.history

    response = client. chat. completions.create(
      model=MODEL,
      messages=messages,
      tools=weather_tool_schema,
      tool_choice = "auto"
    )

    msg = response.choices[0].message
    finish_reason = response.choices[0].finish_reason

    if finish_reason == "tool_calls" and msg.tool_calls:
      tool_call = msg.tool_calls[0]
      tool_args = json.loads(tool_call.function.arguments)
      tool_result = get_weather(**tool_args)

      print(f"[Tool called: get_weather({tool_args})]")

      self.history.append(msg)
      self.history. append({
          "role": "tool",
          "tool_call_id": tool_call.id,
          "content" : json.dumps(tool_result)
      })

      messages = [{"role": "system", "content": self.system}] + self.history

      final = client.chat.completions.create(
        model=MODEL,
        messages=messages
      )
      reply = final.choices[0].message.content
      self.history.append({"role": "assistant", "content": reply})
    else:
      reply = msg.content
      self.history.append({"role": "assistant", "content": reply})

    print(f"Agent : {reply}")
    print(f"[Memory: {len(self.history)} messages stored]")
    return reply

bot = WeatherChatbot()
bot.chat("Hi I live in Raipur")
bot.chat("What's the weather like today?")
bot.chat("Should i take an umbrella when i go out?")


 You : Hi I live in Raipur
Agent : I'll remember you live in Raipur. How can I assist you further about Raipur, or would you like to know something else?
[Memory: 2 messages stored]

 You : What's the weather like today?


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=get_weather>{"city": "Raipur"}'}}